# Compose transformations and inspect the result

Start with a loop adding 0 through 9. Remove a redundant `+ 0`, split the loop into ordered segments, and optionally unroll it. Inspect real before/after IR and see malformed changes refused.

Use the [shared launcher](README.md#open-the-notebooks), then choose **Kernel → Restart Kernel and Run All Cells**. Widgets are optional, and no board or target compiler is needed.

This example inspects proposed rewrites and their structural checks. It does not supply a formal proof or run compiled code. The [transformation guide](../../docs/guides/05-transformations.md) explains the full compiler path.


**Step 4 of 5: Transformations** | [All tutorials](README.md) | [Previous: Scheduling](03_scheduling.ipynb) | [Next: CCA](05_cca.ipynb)


In [ ]:
from html import escape

from xdsl.dialects import arith
from xdsl.dialects.builtin import StringAttr
from xdsl.rewriter import Rewriter

from merlin.transforms.certificates import ir_digest, ir_text, make_change, parse_ir
from merlin.transforms.loops import iter_segments
from merlin.transforms.pipeline import PipelineOptions, apply_common_pipeline
from merlin.transforms.pipeline_state import PassSpec, PipelineContext, PipelineSession
from merlin.transforms.validation import validate_change

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None


def show_html(markup, text):
    if HTML is None:
        print(text)
    else:
        display(HTML(markup))


## 1. Keep the original source

The function adds the indices 0 through 9 into an `i32` accumulator in order. The extra `+ 0` gives the scalar simplifier useful work. The index width is selected separately from the accumulator width.

No transform here needs hardware geometry. `PipelineContext` therefore leaves hardware/resource identities unknown. The pipeline clones the source so each attempt starts from the same program.


In [ ]:
SOURCE = """builtin.module {
  func.func @ordered_sum() -> i32 {
    %lower = arith.constant 0 : index
    %upper = arith.constant 10 : index
    %step = arith.constant 1 : index
    %zero = arith.constant 0 : i32
    %result = scf.for %i = %lower to %upper step %step iter_args(%acc = %zero) -> (i32) {
      %term = arith.index_cast %i : index to i32
      %identity = arith.addi %term, %zero : i32
      %next = arith.addi %acc, %identity : i32
      scf.yield %next : i32
    }
    func.return %result : i32
  }
}"""
source = parse_ir(SOURCE)
source_sha256 = ir_digest(source)
context = PipelineContext()


def transform(tile=3, unroll=2, index_bits=64, simplify=True, plan_only=True):
    options = PipelineOptions(
        index_bits=index_bits,
        normalize_shapes=False,
        simplify_integers=simplify,
        reuse_buffers=False,
        tile_iterations=(tile,),
        unroll_factors=() if unroll == 0 else (unroll,),
    )
    result = apply_common_pipeline(source, options, context=context, plan_only=plan_only)
    assert ir_digest(source) == source_sha256  # The public pipeline clones the frozen input.
    for change in result.changes:
        validate_change(change)
    return result


planned = transform()
assert planned.plan_only
print("Frozen source SHA256:", source_sha256)
print("Structurally checked changes:", [change.kind for change in planned.changes])


## 2. Inspect the actual changes

The segmenter forwards the accumulator through both loops, including the short last segment. Unrolling expands the body while retaining addition order. `validate_change` independently checks the recorded before/after structure; these records are not formal proof receipts.

The picture uses the segmenter's witness and the existing `iter_segments` helper to show index coverage, **not time**. The panes contain real xDSL IR, and the table lists actual executed passes.


In [ ]:
def view_result(result):
    rows = "".join(
        f"<tr><td>{escape(record.name)}</td><td>{record.before.source_sha256[:8]} -> "
        f"{record.after.source_sha256[:8]}</td><td>{escape(', '.join(record.invalidated) or '-')}</td></tr>"
        for record in result.passes
    )
    strip = []
    segmented = next((c for c in result.changes if c.kind == "ordered_loop_segmentation"), None)
    if segmented is not None:
        witness = segmented.certificate.witnesses[0]
        segments = list(iter_segments(witness.lower, witness.upper, witness.step, witness.tile_iterations))
        strip = ['<svg viewBox="0 0 900 65" role="img" aria-label="Ordered index coverage by segment">']
        for i, segment in enumerate(segments):
            x = 10 + 880 * (segment.lower - witness.lower) / (witness.upper - witness.lower)
            width = 880 * (segment.upper - segment.lower) / (witness.upper - witness.lower)
            color = ("#bfdbfe", "#d1fae5")[i % 2]
            strip += [
                f'<rect x="{x}" y="5" width="{width}" height="35" fill="{color}" stroke="white"/>',
                f'<text x="{x + 5}" y="28" fill="#111827">[{segment.lower}, {segment.upper})</text>',
            ]
        strip += ['<text x="10" y="60" fill="#111827">Ordered index coverage; not a device timeline</text></svg>']
    before, after = ir_text(source), ir_text(result.module)
    markup = (
        "".join(strip) + '<div style="display:flex;gap:1em;flex-wrap:wrap">'
        f'<div style="flex:1;min-width:280px"><h4>Original IR</h4><pre style="max-height:330px;overflow:auto">{escape(before)}</pre></div>'
        f'<div style="flex:1;min-width:280px"><h4>Planned IR</h4><pre style="max-height:330px;overflow:auto">{escape(after)}</pre></div></div>'
        "<table><tr><th>Actual pass</th><th>Before -> after digest</th><th>Invalidated analyses</th></tr>"
        f"{rows}</table><p>Planning only. Structural checks completed; host formal admission and compiled execution are not run.</p>"
    )
    show_html(
        markup,
        f"ORIGINAL\n{before}\nPLANNED\n{after}\nPlanning only; {len(result.changes)} structural changes checked.",
    )


view_result(planned)


## 3. Add a small analysis pass

`PipelineSession` keeps analyses tied to the current IR. `PassSpec` names a pass, its stage, what it produces and whether it changes IR. Here we add a static operation count through that existing API.

A later edit must make the count stale. The cell checks this behavior; bookkeeping alone does not prove a rewrite correct.


In [ ]:
analysis_session = PipelineSession(planned.module.clone(), context=context)
summary_spec = PassSpec("static_operation_count", stage=0, produces=("operation_count",), mutates_ir=False)
analysis_session.run(summary_spec, lambda: {"operation_count": sum(1 for _ in analysis_session.module.walk())})
print("Current planned static operations:", analysis_session.analysis("operation_count"))

# A subsequent out-of-band IR edit must not retain the old analysis.
analysis_session.module.attributes["tutorial_revision"] = StringAttr("changed")
try:
    analysis_session.analysis("operation_count")
except ValueError as error:
    print("REFUSED stale analysis:", error)
else:
    raise AssertionError("stale analysis was retained")


## 4. Reject an incorrect rewrite

Replace the segmenter's clipped upper bound, `min`, with `max`. The altered IR still verifies, and we rebuild its hashes. The independent recognizer must therefore reject the changed structure, not just an outdated checksum. We do not execute the altered program.

The second check asks the ordinary pipeline to accept a change without its host verifier. The expected refusal shows that inspecting a plan does not make it a formally accepted result.


In [ ]:
segmentation = next(change for change in planned.changes if change.kind == "ordered_loop_segmentation")
bad_module = parse_ir(segmentation.after_ir)
minimum = next(op for op in bad_module.walk() if isinstance(op, arith.MinUIOp))
Rewriter.replace_op(minimum, arith.MaxUIOp(minimum.lhs, minimum.rhs))
bad_module.verify()
altered = make_change(
    segmentation.kind,
    segmentation.before_ir,
    bad_module,
    segmentation.parameters,
    segmentation.certificate.witnesses,
    segmentation.certificate.scope,
)
altered.validate_identity()
try:
    validate_change(altered)
except ValueError as error:
    print("REFUSED actual changed clipping:", error)
else:
    raise AssertionError("changed clipping passed the independent recognizer")

try:
    transform(plan_only=False)
except ValueError as error:
    print("REFUSED admission without host completion:", error)
else:
    raise AssertionError("planning output was admitted without host completion")


## 5. Compose different choices

Try a segment size of 4 or 16, disable simplification, or turn unrolling off. A four-bit signed index domain is too small for this source's generated index arithmetic and must be refused.

The controls call the same public pipeline against the frozen original. Without widgets, call `explore(tile=4, unroll=0)` directly. Code size and index coverage are compiler observations, not performance measurements.


In [ ]:
def explore(tile=3, unroll=2, index_bits=64, simplify=True):
    try:
        result = transform(tile, unroll, index_bits, simplify)
    except ValueError as error:
        print("REFUSED:", error)
        return
    view_result(result)


explore()
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional controls unavailable; call explore(tile=4, unroll=0).")
else:
    controls = {
        "tile": widgets.Dropdown(options=[2, 3, 4, 16], value=3, description="Segment"),
        "unroll": widgets.Dropdown(
            options=[("None", 0), ("Factor 2", 2), ("Factor 4", 4)], value=2, description="Unroll"
        ),
        "index_bits": widgets.Dropdown(options=[4, 8, 64], value=64, description="Index bits"),
        "simplify": widgets.Checkbox(value=True, description="Simplify integers"),
    }
    output = widgets.interactive_output(explore, controls)
    display(widgets.VBox([widgets.HBox(list(controls.values())), output]))


## Find the next algorithm

| Work | Actual owner | Backend input |
|---|---|---|
| Ordered loop segmentation / unrolling | `transforms/loops.py`, `transforms/unrolling.py` | Supported index domain |
| Block tiling and operand retention | `compiler/schedule/block.py`, `staged_rhs_residency.py` | Geometry, load/compute effects and valid storage |
| Transfer merging | `compiler/schedule/tile_load_coalescing.py` | Address mapping, supported width and completion behavior |
| Command ordering | `compiler/schedule/completion_streams.py` | Complete effects and event/dependency rules |
| Optimization strategy selection | `compiler/optimizations/registry.py` | Applicable policy and actual implementation |

An **out-of-tree (OOT) backend** supplies target-specific effects, physical mapping and encoding from a separate package/repository. Shared passes use those inputs; they do not invent hardware facts.

Continue with the [scheduling notebook](03_scheduling.ipynb) or [transformation guide](../../docs/guides/05-transformations.md). Agents can compose, modify or add passes within the selected source boundaries. Independent proof, numerical/ABI/memory checks and observed device use remain separate from this structural example. Native MLIR/xDSL passes keep their existing pass managers.


**Step 4 of 5: Transformations** | [All tutorials](README.md) | [Previous: Scheduling](03_scheduling.ipynb) | [Next: CCA](05_cca.ipynb)
